# LeetCode 3032 — Count Numbers With Unique Digits II bằng Python

Link đề: https://leetcode.com/problems/count-numbers-with-unique-digits-ii/ · Độ khó: **Easy** · Chủ đề: Math, Hash Table, Dynamic Programming, Enumeration (**bài Premium**)

Skeleton LeetCode:

```python
class Solution:
    def numberCount(self, a: int, b: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium** trên LeetCode. Đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi ví dụ và đáp án trong notebook đã được **kiểm chứng bằng code** (vét cạn), nhưng nên **đối chiếu lại với đề gốc nếu bạn có tài khoản Premium** (đặc biệt phần ràng buộc và các ví dụ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm các số nguyên trong đoạn **`[a, b]` (gồm cả hai đầu)** mà **mỗi chữ số xuất hiện không quá một lần** (các chữ số **đôi một khác nhau**).
2. Biết **tách chữ số** của một số bằng chuyển sang chuỗi (`str`) hoặc bằng **phép chia dư `% 10` và chia nguyên `// 10`**, và kiểm tra trùng lặp bằng `set`, `list` hoặc **bitmask**.
3. Biết dùng `range(a, b + 1)` (nhớ **`+ 1`** để lấy cả `b`) và **mảng cộng dồn (prefix count)** để trả lời nhiều truy vấn đoạn nhanh.
4. Đi từ cách **chuỗi + `set`** → **tách chữ số bằng `%` và `//` với `set` các chữ số đã gặp** → **bitmask 10 bit** → **mảng đếm cộng dồn tính một lần, mỗi truy vấn `O(1)`**.
5. Tránh các lỗi kinh điển: `range(a, b)` làm mất `b`, chỉ so sánh các chữ số **liền kề**, quên đặt lại tập chữ số đã gặp cho từng số, và ghi đè bitmask (`=` thay vì `|=`).

## 2. Đề bài & ví dụ

Cho hai số nguyên **dương** `a` và `b`. Hãy trả về **số lượng các số nguyên trong đoạn `[a, b]`** (gồm cả `a` và `b`) có **các chữ số đôi một khác nhau**, tức **mỗi chữ số xuất hiện không quá một lần** trong số đó.

Ví dụ: `121` **không** thỏa (chữ số `1` xuất hiện hai lần), `123` thỏa, `100` **không** thỏa (chữ số `0` hai lần), `7` thỏa.

**Ví dụ 1**

- Input: `a = 1`, `b = 20`
- Output: `19`
- Giải thích: trong `1, 2, ..., 20`, chỉ có `11` bị loại (hai chữ số `1`). Còn lại `19` số.

**Ví dụ 2**

- Input: `a = 9`, `b = 19`
- Output: `10`
- Giải thích: trong `9, 10, ..., 19`: bị loại duy nhất `11`. Còn lại `11 - 1 = 10` số (đoạn có `11` số).

**Ví dụ 3**

- Input: `a = 80`, `b = 120`
- Output: `27`
- Giải thích: từ `80` đến `99` có `20` số, loại `88` và `99` nên còn `18`. Từ `100` đến `120` có `21` số: `100`, `101` và mọi số `110..119` đều lặp chữ số; chỉ `102, 103, ..., 109` (`8` số) và `120` thỏa, tức `9` số. Tổng `18 + 9 = 27`.

**Ràng buộc:** `1 <= a <= b <= 1000`.

In [1]:
from itertools import accumulate
import random

examples = [
    (1, 20, 19),
    (9, 19, 10),
    (80, 120, 27),
    (1, 9, 9),
    (11, 11, 0),
]
for a, b, expected in examples:
    print(f"a = {a}, b = {b} -> kỳ vọng {expected}")

a = 1, b = 20 -> kỳ vọng 19
a = 9, b = 19 -> kỳ vọng 10
a = 80, b = 120 -> kỳ vọng 27
a = 1, b = 9 -> kỳ vọng 9
a = 11, b = 11 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `range(a, b + 1)`: lấy cả `b`

`range(a, b)` chạy `a, a + 1, ..., b - 1`, **không** gồm `b`. Đoạn **đóng** `[a, b]` cần `range(a, b + 1)`. Đây là lỗi rất hay gặp.

In [2]:
print(list(range(3, 7)))        # [3, 4, 5, 6]     <- không có 7
print(list(range(3, 7 + 1)))    # [3, 4, 5, 6, 7]  <- đoạn đóng [3, 7]
print(len(range(3, 7 + 1)), 7 - 3 + 1)    # 5 5: số phần tử của đoạn [a, b] là b - a + 1

[3, 4, 5, 6]
[3, 4, 5, 6, 7]
5 5


### 3.2. Chuyển số thành chuỗi và `set`

`str(x)` cho chuỗi các chữ số. `set(str(x))` giữ **các chữ số khác nhau**. Một số có các chữ số đôi một khác nhau **khi và chỉ khi** `len(set(str(x))) == len(str(x))` (không có chữ số nào bị loại vì trùng).

In [3]:
for x in [7, 11, 121, 123, 100, 1000]:
    s = str(x)
    print(f"{x:>4}: {s!r:7} set={sorted(set(s))} -> {'các chữ số khác nhau' if len(set(s)) == len(s) else 'có chữ số lặp'}")

   7: '7'     set=['7'] -> các chữ số khác nhau
  11: '11'    set=['1'] -> có chữ số lặp
 121: '121'   set=['1', '2'] -> có chữ số lặp
 123: '123'   set=['1', '2', '3'] -> các chữ số khác nhau
 100: '100'   set=['0', '1'] -> có chữ số lặp
1000: '1000'  set=['0', '1'] -> có chữ số lặp


### 3.3. Tách chữ số bằng `% 10` và `// 10`

`x % 10` là **chữ số hàng đơn vị**, `x // 10` là số sau khi **bỏ** chữ số đó. Lặp `while x > 0` ta lấy lần lượt các chữ số từ **phải sang trái**, không cần chuyển sang chuỗi.

In [4]:
x = 1204
while x > 0:
    print("chữ số:", x % 10, "| còn lại:", x // 10)
    x //= 10

chữ số: 4 | còn lại: 120
chữ số: 0 | còn lại: 12
chữ số: 2 | còn lại: 1
chữ số: 1 | còn lại: 0


### 3.4. Theo dõi chữ số đã gặp bằng `set`

Lập `seen = set()`. Với mỗi chữ số `d`: nếu `d in seen` thì **trùng** (dừng ngay); ngược lại `seen.add(d)`. Mỗi số phải có `seen` **mới** (đặt lại) khi xét số khác.

In [5]:
def has_unique_digits(x):
    seen = set()
    while x > 0:
        d = x % 10
        if d in seen:
            return False                  # chữ số lặp
        seen.add(d)
        x //= 10
    return True

print([has_unique_digits(v) for v in (7, 11, 121, 123, 100)])    # [True, False, False, True, False]

[True, False, False, True, False]


### 3.5. Bitmask 10 bit cho 10 chữ số

Thay `set` bằng **một số nguyên** `mask` có 10 bit; bit `d` bằng `1` nghĩa là chữ số `d` đã gặp. Kiểm tra gặp rồi: `mask >> d & 1` (hay `mask & (1 << d)`); đánh dấu: `mask |= 1 << d`. Nhanh và không cấp phát bộ nhớ.

In [6]:
mask = 0
for d in [1, 2, 0]:
    print(f"d={d}: đã gặp? {mask >> d & 1 == 1}")
    mask |= 1 << d                         # |= : BẬT thêm bit d, giữ các bit cũ
    print("   mask =", bin(mask))

d=1: đã gặp? False
   mask = 0b10
d=2: đã gặp? False
   mask = 0b110
d=0: đã gặp? False
   mask = 0b111


### 3.6. Mảng cộng dồn (prefix count) cho truy vấn đoạn

Nếu `pre[x]` là số số hợp lệ trong `1..x` (`pre[0] = 0`) thì số số hợp lệ trong `[a, b]` là **`pre[b] - pre[a - 1]`**. Tính `pre` **một lần** cho mọi `x` đến `1000`, mỗi truy vấn sau đó `O(1)`. `itertools.accumulate` giúp cộng dồn.

In [7]:
good = [0] + [1 if has_unique_digits(x) else 0 for x in range(1, 21)]      # good[x] = 1 nếu x hợp lệ
pre = list(accumulate(good))                                               # pre[x] = số hợp lệ trong 1..x
print(pre)
print("[9, 19]:", pre[19] - pre[8])        # 10
print("[1, 20]:", pre[20] - pre[0])        # 19

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[9, 19]: 10
[1, 20]: 19


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Duyệt từng số trong `[a, b]`, **kiểm tra riêng từng số**: có chữ số nào lặp không? Với `a = 9`, `b = 19`:

| `x` | chữ số | lặp? | hợp lệ? |
|---|---|---|---|
| 9 | `9` | không | có |
| 10 | `1, 0` | không | có |
| 11 | `1, 1` | **có** | không |
| 12 | `1, 2` | không | có |
| 13 | `1, 3` | không | có |
| 14 | `1, 4` | không | có |
| 15 | `1, 5` | không | có |
| 16 | `1, 6` | không | có |
| 17 | `1, 7` | không | có |
| 18 | `1, 8` | không | có |
| 19 | `1, 9` | không | có |

Có `11` số, trừ `11` còn `10` hợp lệ: khớp Ví dụ 2. Với ràng buộc `b <= 1000` mỗi số tối đa `4` chữ số, nên cách "kiểm tra từng số" hoàn toàn đủ nhanh.

In [8]:
a, b = 9, 19
count = 0
for x in range(a, b + 1):
    s = str(x)
    ok = len(set(s)) == len(s)
    count += ok
    print(f"x={x:>2} chữ số={list(s)} -> {'hợp lệ' if ok else 'LẶP'} | count={count}")

x= 9 chữ số=['9'] -> hợp lệ | count=1
x=10 chữ số=['1', '0'] -> hợp lệ | count=2
x=11 chữ số=['1', '1'] -> LẶP | count=2
x=12 chữ số=['1', '2'] -> hợp lệ | count=3
x=13 chữ số=['1', '3'] -> hợp lệ | count=4
x=14 chữ số=['1', '4'] -> hợp lệ | count=5
x=15 chữ số=['1', '5'] -> hợp lệ | count=6
x=16 chữ số=['1', '6'] -> hợp lệ | count=7
x=17 chữ số=['1', '7'] -> hợp lệ | count=8
x=18 chữ số=['1', '8'] -> hợp lệ | count=9
x=19 chữ số=['1', '9'] -> hợp lệ | count=10


## 5. Cách 1 — Brute Force / baseline: chuỗi + `set` cho từng số

**Ý tưởng.** Với mỗi `x` trong `range(a, b + 1)`: chuyển thành chuỗi, so sánh `len(set(s))` với `len(s)`; bằng nhau thì đếm.

In [9]:
def number_count_str(a: int, b: int) -> int:
    count = 0
    for x in range(a, b + 1):                         # b + 1: lấy cả b
        s = str(x)
        if len(set(s)) == len(s):                     # không có chữ số nào bị loại vì trùng
            count += 1
    return count


for a, b, expected in examples:
    print(a, b, "->", number_count_str(a, b), "(kỳ vọng", expected, ")")

1 20 -> 19 (kỳ vọng 19 )
9 19 -> 10 (kỳ vọng 10 )
80 120 -> 27 (kỳ vọng 27 )
1 9 -> 9 (kỳ vọng 9 )
11 11 -> 0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Có `b - a + 1 <= 1000` số, mỗi số tối đa `4` chữ số: **Time `O((b - a + 1) · d)`** với `d` là số chữ số (`<= 4`), **Space `O(d)`**.

**Nút thắt.** Với ràng buộc nhỏ thì đã quá đủ. Nhưng việc chuyển sang chuỗi tạo đối tượng mới mỗi lần, và nếu có **nhiều truy vấn** `(a, b)` ta sẽ lặp lại cùng việc kiểm tra các số nhiều lần.

## 6. Ý tưởng thuật toán / bước đột phá

Hai hướng cải tiến, mỗi hướng cho một mục đích:

1. **Bỏ chuỗi, tách chữ số bằng `%`/`//`** và theo dõi chữ số đã gặp bằng `set` rồi **bitmask**: giảm việc cấp phát và dừng sớm khi gặp chữ số trùng.
2. **Tách "kiểm tra một số" khỏi "đếm trên đoạn"**: ai là số hợp lệ **không phụ thuộc** `a`, `b`. Tính một lần `good[x]` cho `x = 1..1000`, rồi mảng cộng dồn `pre`; mỗi truy vấn `(a, b)` là **`pre[b] - pre[a - 1]`** (`O(1)`).

Hướng 2 là mẫu **tiền xử lý rồi trả lời truy vấn đoạn** cực kỳ phổ biến.

## 7. Các cách tiếp theo

### Cách 2 — Tách chữ số bằng `%` và `//`, theo dõi bằng `set`

**Giải thích.** Viết hàm phụ `unique_digits(x)` như mục 3.4: lặp `while x > 0`, lấy `d = x % 10`; nếu `d` đã có trong `seen` thì trả `False`, ngược lại thêm vào. Đếm các `x` thỏa.

In [10]:
def unique_digits_set(x: int) -> bool:
    seen = set()                                      # mới cho MỖI số
    while x > 0:
        d = x % 10
        if d in seen:
            return False
        seen.add(d)
        x //= 10
    return True


def number_count_digits(a: int, b: int) -> int:
    return sum(1 for x in range(a, b + 1) if unique_digits_set(x))


for a, b, expected in examples:
    print(a, b, "->", number_count_digits(a, b), "(kỳ vọng", expected, ")")

1 20 -> 19 (kỳ vọng 19 )
9 19 -> 10 (kỳ vọng 10 )
80 120 -> 27 (kỳ vọng 27 )
1 9 -> 9 (kỳ vọng 9 )
11 11 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O((b - a + 1) · d)`**, **Space `O(1)`** (`set` tối đa `10` phần tử).

### Cách 3 — Bitmask 10 bit

**Giải thích.** Thay `set` bằng một số nguyên `mask`: bit `d` bật khi chữ số `d` đã xuất hiện. `mask |= 1 << d` để đánh dấu (nhớ `|=`, **không** phải `=`). Không cấp phát đối tượng nào, thường nhanh nhất cho phần kiểm tra từng số.

In [11]:
def unique_digits_mask(x: int) -> bool:
    mask = 0                                          # mới cho MỖI số
    while x > 0:
        d = x % 10
        if mask >> d & 1:                             # bit d đã bật: chữ số lặp
            return False
        mask |= 1 << d                                # bật bit d, giữ các bit cũ
        x //= 10
    return True


def number_count_mask(a: int, b: int) -> int:
    count = 0
    for x in range(a, b + 1):
        if unique_digits_mask(x):
            count += 1
    return count


for a, b, expected in examples:
    print(a, b, "->", number_count_mask(a, b), "(kỳ vọng", expected, ")")

1 20 -> 19 (kỳ vọng 19 )
9 19 -> 10 (kỳ vọng 10 )
80 120 -> 27 (kỳ vọng 27 )
1 9 -> 9 (kỳ vọng 9 )
11 11 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 3:** **Time `O((b - a + 1) · d)`**, **Space `O(1)`** (một số nguyên làm mặt nạ).

### Cách 4 — Tiền xử lý một lần + mảng cộng dồn (`O(1)` mỗi truy vấn)

**Giải thích.** Tính `good[x] = 1` nếu `x` hợp lệ, `pre[x] = good[1] + ... + good[x]` cho `x = 0..1000` (`pre[0] = 0`). Số hợp lệ trong `[a, b]` là `pre[b] - pre[a - 1]`. Chi phí `O(1000)` một lần, sau đó **mỗi truy vấn `O(1)`**. Hợp với bài toán có **rất nhiều truy vấn**.

In [12]:
MAX = 1000
good = [0] * (MAX + 1)
for x in range(1, MAX + 1):
    good[x] = 1 if unique_digits_mask(x) else 0
pre = list(accumulate(good))                          # pre[x] = số số hợp lệ trong [1, x]; pre[0] = 0


def number_count_prefix(a: int, b: int) -> int:
    return pre[b] - pre[a - 1]


for a, b, expected in examples:
    print(a, b, "->", number_count_prefix(a, b), "(kỳ vọng", expected, ")")

print("pre[9], pre[99], pre[999], pre[1000] =", pre[9], pre[99], pre[999], pre[1000])    # 9 90 738 738

1 20 -> 19 (kỳ vọng 19 )
9 19 -> 10 (kỳ vọng 10 )
80 120 -> 27 (kỳ vọng 27 )
1 9 -> 9 (kỳ vọng 9 )
11 11 -> 0 (kỳ vọng 0 )
pre[9], pre[99], pre[999], pre[1000] = 9 90 738 738


**Độ phức tạp Cách 4:** tiền xử lý **Time `O(V · d)`** (`V = 1000`), **Space `O(V)`**; mỗi truy vấn **`O(1)`**.

**Kiểm tra bằng đếm tổ hợp.** Số có đúng `k` chữ số và các chữ số đôi một khác nhau: `k = 1`: `9`; `k = 2`: `9 · 9 = 81` (hàng chục `1..9`, hàng đơn vị `0..9` trừ chữ số đã dùng); `k = 3`: `9 · 9 · 8 = 648`. Tổng `1..999`: `9 + 81 + 648 = 738`, và `1000` có ba chữ số `0` lặp nên bị loại, đúng với `pre[1000] = 738`.

In [13]:
# Code nộp LeetCode cho Cách 3 (đơn giản, chạy tức thì với b <= 1000)
class Solution:
    def numberCount(self, a: int, b: int) -> int:
        count = 0
        for x in range(a, b + 1):
            mask = 0
            ok = True
            while x > 0:
                d = x % 10
                if mask >> d & 1:
                    ok = False
                    break
                mask |= 1 << d
                x //= 10
            if ok:
                count += 1
        return count

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 (bitmask) với `x = 1201` và `x = 1230`, và (b) Cách 4 (cộng dồn) với `[80, 120]`.

In [14]:
def trace_mask(x):
    original = x
    mask = 0
    print(f"x = {original}")
    while x > 0:
        d = x % 10
        seen = mask >> d & 1
        print(f"  chữ số {d}: bit {d} trước = {seen} | mask trước = {mask:010b}", end="")
        if seen:
            print(" -> LẶP, kết luận: không hợp lệ")
            return False
        mask |= 1 << d
        print(f" -> mask sau = {mask:010b}")
        x //= 10
    print("  hết chữ số -> hợp lệ")
    return True


assert trace_mask(1230) is True
assert trace_mask(1201) is False
assert trace_mask(88) is False

x = 1230
  chữ số 0: bit 0 trước = 0 | mask trước = 0000000000 -> mask sau = 0000000001
  chữ số 3: bit 3 trước = 0 | mask trước = 0000000001 -> mask sau = 0000001001
  chữ số 2: bit 2 trước = 0 | mask trước = 0000001001 -> mask sau = 0000001101
  chữ số 1: bit 1 trước = 0 | mask trước = 0000001101 -> mask sau = 0000001111
  hết chữ số -> hợp lệ
x = 1201
  chữ số 1: bit 1 trước = 0 | mask trước = 0000000000 -> mask sau = 0000000010
  chữ số 0: bit 0 trước = 0 | mask trước = 0000000010 -> mask sau = 0000000011
  chữ số 2: bit 2 trước = 0 | mask trước = 0000000011 -> mask sau = 0000000111
  chữ số 1: bit 1 trước = 1 | mask trước = 0000000111 -> LẶP, kết luận: không hợp lệ
x = 88
  chữ số 8: bit 8 trước = 0 | mask trước = 0000000000 -> mask sau = 0100000000
  chữ số 8: bit 8 trước = 1 | mask trước = 0100000000 -> LẶP, kết luận: không hợp lệ


In [15]:
def trace_prefix(a, b):
    print(f"[{a}, {b}]: pre[{b}] = {pre[b]}, pre[{a - 1}] = {pre[a - 1]} -> {pre[b]} - {pre[a - 1]} = {pre[b] - pre[a - 1]}")
    return pre[b] - pre[a - 1]


assert trace_prefix(80, 120) == 27
assert trace_prefix(9, 19) == 10
assert trace_prefix(1, 1000) == 738

[80, 120]: pre[120] = 99, pre[79] = 72 -> 99 - 72 = 27
[9, 19]: pre[19] = 18, pre[8] = 8 -> 18 - 8 = 10
[1, 1000]: pre[1000] = 738, pre[0] = 0 -> 738 - 0 = 738


Bảng tóm tắt (Cách 4):

| `a, b` | `pre[b]` | `pre[a - 1]` | đáp án |
|---|---|---|---|
| `1, 20` | `pre[20] = 19` | `pre[0] = 0` | `19` |
| `9, 19` | `pre[19] = 18` | `pre[8] = 8` | `10` |
| `80, 120` | `pre[120]` | `pre[79]` | `27` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (bitmask, tính trực tiếp từng số). Đúng class `Solution`, đúng method `numberCount(self, a: int, b: int) -> int`.

In [16]:
class Solution:
    def numberCount(self, a: int, b: int) -> int:
        count = 0
        for x in range(a, b + 1):                     # b + 1: lấy cả b
            mask = 0                                  # bit d = 1 nếu chữ số d đã xuất hiện
            ok = True
            while x > 0:
                d = x % 10                            # chữ số hàng đơn vị
                if mask >> d & 1:                     # chữ số d đã gặp: lặp
                    ok = False
                    break
                mask |= 1 << d                        # đánh dấu đã gặp d (|=, giữ các bit cũ)
                x //= 10                              # bỏ chữ số vừa xét
            if ok:
                count += 1
        return count


sol = Solution()
for a, b, expected in examples:
    print(f"a={a}, b={b} ->", sol.numberCount(a, b))

a=1, b=20 -> 19
a=9, b=19 -> 10
a=80, b=120 -> 27
a=1, b=9 -> 9
a=11, b=11 -> 0


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa.** Một số có các chữ số đôi một khác nhau ⇔ không có chữ số nào xuất hiện hai lần ⇔ khi duyệt chữ số, không bao giờ gặp một chữ số đã có. Bitmask `mask` ghi chính xác tập chữ số đã gặp, nên kiểm tra `mask >> d & 1` đúng khi và chỉ khi `d` đã xuất hiện.
- **Tách chữ số đủ.** Với `x > 0`, vòng `while x > 0` lấy lần lượt chữ số hàng đơn vị rồi bỏ nó, nên duyệt **đủ và đúng một lần** mọi chữ số (kể cả chữ số `0` ở giữa, vì điều kiện là `x > 0` chứ không phải `d > 0`). Thứ tự phải-sang-trái không ảnh hưởng đến việc có lặp hay không.
- **Đủ đoạn.** `range(a, b + 1)` đi qua đúng các số `a, ..., b`.
- **Cách 4 đúng.** `pre[x] = Σ_{t ≤ x} good[t]` nên `pre[b] - pre[a - 1] = Σ_{a ≤ t ≤ b} good[t]`: đúng số hợp lệ trong `[a, b]`. Vì `a >= 1` nên `a - 1 >= 0` là chỉ số hợp lệ.
- **Kiểm tra tổ hợp độc lập.** Đếm theo số chữ số cho `9 + 81 + 648 = 738` số hợp lệ trong `1..999`, khớp với kết quả vét cạn.

## 11. Phân tích Time / Space Complexity

Gọi `m = b - a + 1` là số phần tử của đoạn, `d <= 4` là số chữ số, `V = 1000`.

- **Cách 1 (chuỗi + `set`):** Time `O(m · d)`, Space `O(d)`.
- **Cách 2 (`%`/`//` + `set`):** Time `O(m · d)`, Space `O(1)`.
- **Cách 3 (bitmask):** Time `O(m · d)`, Space `O(1)`.
- **Cách 4 (tiền xử lý + cộng dồn):** tiền xử lý `O(V · d)`, Space `O(V)`; mỗi truy vấn `O(1)`.
- Với `m <= 1000` và `d <= 4` mọi cách đều chạy tức thì (`~4 000` phép).

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "str": number_count_str,
    "digits": number_count_digits,
    "mask": number_count_mask,
    "prefix": number_count_prefix,
    "submit": sol.numberCount,
}

for a, b, expected in examples:
    for key, fn in impls.items():
        got = fn(a, b)
        assert got == expected, (key, a, b, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(a, b, expected, label):
    for key, fn in impls.items():
        got = fn(a, b)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all(1, 1, 1, "một số một chữ số")
check_all(5, 5, 1, "một số một chữ số khác")
check_all(11, 11, 0, "một số hai chữ số lặp")
check_all(10, 10, 1, "10 có chữ số 1 và 0 khác nhau")
check_all(100, 100, 0, "100 có hai chữ số 0")
check_all(101, 101, 0, "101: chữ số 1 lặp nhưng KHÔNG liền kề")
check_all(121, 121, 0, "121: lặp không liền kề")
check_all(123, 123, 1, "123 hợp lệ")
check_all(1, 9, 9, "mọi số một chữ số")
check_all(1, 99, 90, "9 + 81 = 90")
check_all(10, 99, 81, "mọi số hai chữ số: 9 * 9")
check_all(100, 999, 648, "mọi số ba chữ số: 9 * 9 * 8")
check_all(1, 1000, 738, "toàn bộ miền ràng buộc: 9 + 81 + 648")
check_all(1000, 1000, 0, "1000: ba chữ số 0 lặp")
check_all(999, 1000, 0, "999 và 1000 đều không hợp lệ")
check_all(98, 102, 2, "98 ok, 99 lặp, 100 lặp, 101 lặp, 102 ok")

PASS một số một chữ số -> 1
PASS một số một chữ số khác -> 1
PASS một số hai chữ số lặp -> 0
PASS 10 có chữ số 1 và 0 khác nhau -> 1
PASS 100 có hai chữ số 0 -> 0
PASS 101: chữ số 1 lặp nhưng KHÔNG liền kề -> 0
PASS 121: lặp không liền kề -> 0
PASS 123 hợp lệ -> 1
PASS mọi số một chữ số -> 9
PASS 9 + 81 = 90 -> 90
PASS mọi số hai chữ số: 9 * 9 -> 81
PASS mọi số ba chữ số: 9 * 9 * 8 -> 648
PASS toàn bộ miền ràng buộc: 9 + 81 + 648 -> 738
PASS 1000: ba chữ số 0 lặp -> 0
PASS 999 và 1000 đều không hợp lệ -> 0
PASS 98 ok, 99 lặp, 100 lặp, 101 lặp, 102 ok -> 2


### 12.2. Tính chất và phản ví dụ

In [19]:
# Chuẩn vàng độc lập: định nghĩa trực tiếp bằng Counter
from collections import Counter

def reference(a, b):
    return sum(1 for x in range(a, b + 1) if max(Counter(str(x)).values()) == 1)


# Duyệt TOÀN BỘ các cặp (a, b) với 1 <= a <= b <= 60 và so khớp mọi cách
for a in range(1, 61):
    for b in range(a, 61):
        ref = reference(a, b)
        for key, fn in impls.items():
            assert fn(a, b) == ref, (key, a, b)
print("PASS mọi cặp (a, b) với 1 <= a <= b <= 60")

# Tính chất 1: cộng được theo đoạn: f(a, b) = f(a, m) + f(m + 1, b)
random.seed(11)
for _ in range(500):
    a = random.randint(1, 999)
    b = random.randint(a + 1, 1000)
    m = random.randint(a, b - 1)
    assert sol.numberCount(a, b) == sol.numberCount(a, m) + sol.numberCount(m + 1, b)
print("PASS f(a, b) = f(a, m) + f(m + 1, b)")

# Tính chất 2: f(a, a) chỉ là 0 hoặc 1; f(a, b) <= b - a + 1; f(a, b) tăng khi mở rộng đoạn
for _ in range(500):
    a = random.randint(1, 1000)
    assert sol.numberCount(a, a) in (0, 1)
    b = random.randint(a, 1000)
    assert 0 <= sol.numberCount(a, b) <= b - a + 1
    if b < 1000:
        assert sol.numberCount(a, b + 1) >= sol.numberCount(a, b)
print("PASS 0 <= f <= độ dài đoạn; đơn điệu khi mở rộng")

# Tính chất 3: theo số chữ số
assert [sol.numberCount(10 ** (k - 1), 10 ** k - 1) for k in (1, 2, 3)] == [9, 81, 648]
print("PASS đếm theo số chữ số: 9, 9*9, 9*9*8")

# Phản ví dụ 1: range(a, b) làm mất b
def sai_range(a, b):
    return sum(1 for x in range(a, b) if len(set(str(x))) == len(str(x)))

assert sai_range(1, 20) == 18 != 19
print("PASS phản ví dụ: range(a, b) cho", sai_range(1, 20), "thay vì 19 ở [1, 20]")

# Phản ví dụ 2: chỉ so sánh chữ số liền kề
def sai_lien_ke(a, b):
    def ok(x):
        s = str(x)
        return all(s[i] != s[i + 1] for i in range(len(s) - 1))
    return sum(1 for x in range(a, b + 1) if ok(x))

assert sai_lien_ke(101, 101) == 1 and sol.numberCount(101, 101) == 0
print("PASS phản ví dụ: chỉ xét liền kề coi 101 là hợp lệ")

PASS mọi cặp (a, b) với 1 <= a <= b <= 60


PASS f(a, b) = f(a, m) + f(m + 1, b)


PASS 0 <= f <= độ dài đoạn; đơn điệu khi mở rộng
PASS đếm theo số chữ số: 9, 9*9, 9*9*8
PASS phản ví dụ: range(a, b) cho 18 thay vì 19 ở [1, 20]
PASS phản ví dụ: chỉ xét liền kề coi 101 là hợp lệ


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(3032)
for trial in range(3000):
    a = random.randint(1, 1000)
    b = random.randint(a, 1000)
    ref = reference(a, b)
    for key, fn in impls.items():
        assert fn(a, b) == ref, (key, a, b)
print("PASS 3000 cặp (a, b) ngẫu nhiên trên miền 1..1000 - mọi cách khớp chuẩn vàng độc lập")

PASS 3000 cặp (a, b) ngẫu nhiên trên miền 1..1000 - mọi cách khớp chuẩn vàng độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `range(a, b)` và làm mất `b`

In [21]:
def sai_range_2(a, b):
    count = 0
    for x in range(a, b):                                # thiếu b
        if len(set(str(x))) == len(str(x)):
            count += 1
    return count


print("sai  :", sai_range_2(1, 20), "| đúng:", Solution().numberCount(1, 20))
print("sai  :", sai_range_2(5, 5), "| đúng:", Solution().numberCount(5, 5))

sai  : 18 | đúng: 19
sai  : 0 | đúng: 1


**Vì sao sai:** `range(a, b)` **không** chứa `b`. Đoạn `[a, b]` có `b - a + 1` số, cần `range(a, b + 1)`. Khi `a == b` lỗi còn nặng hơn: `range(5, 5)` rỗng nên cho `0` thay vì `1`.

### Lỗi 2: Chỉ so sánh các chữ số liền kề

In [22]:
def sai_lien_ke_2(a, b):
    count = 0
    for x in range(a, b + 1):
        s = str(x)
        if all(s[i] != s[i + 1] for i in range(len(s) - 1)):       # chỉ xét hai chữ số cạnh nhau
            count += 1
    return count


print("sai  :", sai_lien_ke_2(101, 101), "| đúng:", Solution().numberCount(101, 101))

sai  : 1 | đúng: 0


**Vì sao sai:** "đôi một khác nhau" nghĩa là **mọi cặp** chữ số, không chỉ cặp kề nhau. `101` có hai chữ số `1` ở hai đầu (không kề nhau) nên **không** thỏa. Dùng `set` hoặc bitmask để kiểm tra mọi cặp.

### Lỗi 3: Không đặt lại tập chữ số đã gặp cho từng số

In [23]:
def sai_khong_reset(a, b):
    seen = set()                                         # đặt NGOÀI vòng lặp các số
    count = 0
    for x in range(a, b + 1):
        ok = True
        y = x
        while y > 0:
            d = y % 10
            if d in seen:
                ok = False
            seen.add(d)
            y //= 10
        if ok:
            count += 1
    return count


print("sai  :", sai_khong_reset(1, 20), "| đúng:", Solution().numberCount(1, 20))

sai  : 9 | đúng: 19


**Vì sao sai:** `seen` giữ các chữ số của **mọi số trước đó**, nên số `10` bị coi là lặp (đã thấy `1` ở số `1`). Tập chữ số đã gặp phải **mới cho từng số** (khởi tạo bên trong vòng lặp theo `x`).

### Lỗi 4: Ghi đè bitmask (`=` thay vì `|=`)

In [24]:
def sai_ghi_de_mask(a, b):
    count = 0
    for x in range(a, b + 1):
        mask, ok, y = 0, True, x
        while y > 0:
            d = y % 10
            if mask >> d & 1:
                ok = False
                break
            mask = 1 << d                                # ghi đè: mất các chữ số đã gặp trước đó
            y //= 10
        if ok:
            count += 1
    return count


print("sai  :", sai_ghi_de_mask(100, 101), "| đúng:", Solution().numberCount(100, 101))

sai  : 1 | đúng: 0


**Vì sao sai:** `mask = 1 << d` xóa mọi bit cũ và chỉ nhớ **chữ số vừa xét**, nên `101` (`1`, `0`, `1`) không bị phát hiện lặp (khi gặp `1` thứ hai thì `mask` chỉ còn bit `0`). Phải **`mask |= 1 << d`** để giữ cả các bit đã có.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đếm số trong `[a, b]` thỏa tính chất về **chữ số**" với `b` nhỏ → **duyệt từng số** và kiểm tra (chuỗi hoặc `%`/`//`).
- Đoạn **đóng** `[a, b]` → `range(a, b + 1)`. Số phần tử `b - a + 1`.
- "Chữ số đôi một khác nhau" → `len(set(str(x))) == len(str(x))` hoặc bitmask `mask |= 1 << d`.
- **Nhiều truy vấn đoạn** → tiền xử lý `good[]` + `accumulate` → `pre[b] - pre[a - 1]`.
- Khi `b` rất lớn (`10¹⁸`) cần **quy hoạch động chữ số (digit DP)** hoặc đếm tổ hợp; đừng duyệt từng số.
- Đếm theo số chữ số: `9`, `9·9`, `9·9·8`, `9·9·8·7`, ... (chữ số đầu `1..9`, các chữ số sau lấy từ những chữ số còn lại).

## 15. Biến thể của bài

1. **Bài 357** (Count Numbers with Unique Digits): đếm số có chữ số đôi một khác nhau trong `[0, 10ⁿ)`.
2. Tìm số hợp lệ **nhỏ nhất lớn hơn `x`**.
3. Tìm số hợp lệ **thứ `k`**.
4. Đếm số có **đúng `k` chữ số khác nhau** (cho phép lặp).
5. Đếm số trong `[a, b]` mà **tổng các chữ số** bằng một giá trị cho trước.

Thử biến thể 1, 2 và 4:

In [25]:
def count_unique_digits_n(n):
    """Bài 357: số có các chữ số đôi một khác nhau trong [0, 10^n)."""
    if n == 0:
        return 1
    total, cur, available = 10, 9, 9
    for _ in range(2, min(n, 10) + 1):
        cur *= available
        available -= 1
        total += cur
    return total


def next_unique(x):
    x += 1
    while not unique_digits_mask(x):
        x += 1
    return x


def count_exactly_k_distinct(a, b, k):
    return sum(1 for x in range(a, b + 1) if len(set(str(x))) == k)


print("1. n = 0..4:", [count_unique_digits_n(n) for n in range(5)])      # [1, 10, 91, 739, 5275]
print("2. số hợp lệ kế tiếp sau 98:", next_unique(98), "| sau 100:", next_unique(100))   # 102, 102
print("4. đúng 2 chữ số khác nhau trong [1, 100]:", count_exactly_k_distinct(1, 100, 2))
assert [count_unique_digits_n(n) for n in range(5)] == [1, 10, 91, 739, 5275]
assert count_unique_digits_n(3) - 1 == Solution().numberCount(1, 999)      # bỏ số 0 vì bài 3032 bắt đầu từ 1
assert count_exactly_k_distinct(1, 100, 2) == 82                            # 81 số hai chữ số khác nhau + số 100
assert next_unique(98) == 102 and next_unique(100) == 102
assert count_exactly_k_distinct(1, 9, 1) == 9

1. n = 0..4: [1, 10, 91, 739, 5275]
2. số hợp lệ kế tiếp sau 98: 102 | sau 100: 102
4. đúng 2 chữ số khác nhau trong [1, 100]: 82


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi + `set` | `len(set(s)) == len(s)` | `O(m·d)` | `O(d)` | dễ viết, dễ hiểu |
| 2. `%`/`//` + `set` | tách chữ số số học | `O(m·d)` | `O(1)` | không muốn dùng chuỗi |
| 3. Bitmask 10 bit | `mask \|= 1 << d` | `O(m·d)` | `O(1)` | **cách nộp bài**, không cấp phát |
| 4. Tiền xử lý + cộng dồn | `pre[b] - pre[a-1]` | `O(V·d)` + `O(1)`/truy vấn | `O(V)` | rất nhiều truy vấn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len(set("1221"))`, `1234 % 10`, `1234 // 10`, `Solution().numberCount(20, 29)`, `Solution().numberCount(100, 110)`.

**Bài 2 (tính tay).** Với `a = 95`, `b = 105`: liệt kê từng số từ `95` đến `105`, đánh dấu số có chữ số lặp và đếm số hợp lệ.

**Bài 3 (điền TODO).** Hoàn thành `number_count_todo(a, b)` theo Cách 1.

In [26]:
def number_count_todo(a, b):
    count = 0
    # TODO 1: duyệt x từ a đến b (nhớ lấy cả b: dùng range(a, b + 1))
    #     TODO 2: s = str(x); nếu len(set(s)) == len(s) thì count += 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `number_count_prefix2(a, b)` theo **Cách 4** (tiền xử lý một lần, `pre[b] - pre[a - 1]`) mà không nhìn code mẫu. Kiểm tra với năm ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kth_unique(k)`: số hợp lệ thứ `k` (`k = 1` là `1`), với `k <= 738`. (b) Viết `longest_run(a, b)`: độ dài **đoạn liên tiếp dài nhất** các số hợp lệ trong `[a, b]`. (c) Viết `count_digit_sum(a, b, s)`: đếm số hợp lệ trong `[a, b]` có **tổng các chữ số bằng `s`**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `4`, `123`, `9` (`20..29` có `10` số, loại `22`), `8` (`100..110`: các số hợp lệ là `102..109`). Bài 3: `for x in range(a, b + 1)`.
- **Mức 2 (vừa):** Bài 2: `95..99`: `95` ok, `96` ok, `97` ok, `98` ok, `99` lặp; `100` lặp, `101` lặp, `102` ok, `103` ok, `104` ok, `105` ok. Đếm hợp lệ: `4 + 4 = 8`. Bài 4: `pre[b] - pre[a - 1]`.
- **Mức 3 (sâu):** Bài 5(a): duyệt `x` tăng dần, đếm số hợp lệ đến khi bằng `k`; hoặc `[x for x in range(1, 1001) if unique_digits_mask(x)][k - 1]`. 5(b): giữ `run` hiện tại, đặt lại về `0` khi gặp số không hợp lệ, cập nhật `best`. 5(c): thêm điều kiện `sum(map(int, str(x))) == s` cho các số hợp lệ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert len(set("1221")) == 2 and 1234 % 10 == 4 and 1234 // 10 == 123
assert Solution().numberCount(20, 29) == 9
assert Solution().numberCount(100, 110) == 8

# Bài 2
assert [x for x in range(95, 106) if len(set(str(x))) == len(str(x))] == [95, 96, 97, 98, 102, 103, 104, 105]
assert Solution().numberCount(95, 105) == 8

# Bài 3
def number_count_todo_ans(a, b):
    count = 0
    for x in range(a, b + 1):
        s = str(x)
        if len(set(s)) == len(s):
            count += 1
    return count

for a, b, expected in examples:
    assert number_count_todo_ans(a, b) == expected

# Bài 4
def number_count_prefix2(a, b):
    n = 1000
    good2 = [0] * (n + 1)
    for x in range(1, n + 1):
        s = str(x)
        good2[x] = 1 if len(set(s)) == len(s) else 0
    pre2 = list(accumulate(good2))
    return pre2[b] - pre2[a - 1]

for a, b, expected in examples:
    assert number_count_prefix2(a, b) == expected

# Bài 5(a)
def kth_unique(k):
    return [x for x in range(1, 1001) if unique_digits_mask(x)][k - 1]

assert kth_unique(1) == 1 and kth_unique(9) == 9 and kth_unique(10) == 10 and kth_unique(11) == 12
assert kth_unique(738) == 987

# Bài 5(b)
def longest_run(a, b):
    best = run = 0
    for x in range(a, b + 1):
        if unique_digits_mask(x):
            run += 1
            best = max(best, run)
        else:
            run = 0
    return best

assert longest_run(1, 10) == 10
assert longest_run(1, 11) == 10
assert longest_run(98, 102) == 1
assert longest_run(102, 109) == 8

# Bài 5(c)
def count_digit_sum(a, b, s):
    return sum(1 for x in range(a, b + 1) if unique_digits_mask(x) and sum(map(int, str(x))) == s)

assert count_digit_sum(1, 99, 3) == 4          # 3, 12, 21, 30
assert count_digit_sum(1, 20, 1) == 2          # 1, 10
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt từng số trong đoạn đóng, kiểm tra chữ số bằng `set`)

```python
class Solution:
    def numberCount(self, a: int, b: int) -> int:
        count = 0
        for x in range(a, b + 1):
            s = str(x)
            if len(set(s)) == len(s):
                count += 1
        return count
```

**Checklist**

- [ ] Dùng `range(a, b + 1)` (lấy cả `b`)?
- [ ] Kiểm tra **mọi cặp** chữ số (`set`/bitmask), không chỉ chữ số liền kề?
- [ ] Tập chữ số đã gặp (hoặc `mask`) được **đặt lại cho từng số**?
- [ ] Nếu dùng bitmask: `mask |= 1 << d` (không phải `=`)?
- [ ] Nếu tách chữ số: `while x > 0` (không `while d > 0`, vì chữ số `0` ở giữa vẫn cần xét)?
- [ ] Đã thử: `a == b`, số hai chữ số lặp (`11`), lặp không liền kề (`101`, `121`), `b = 1000`?
- [ ] Skeleton đúng `class Solution` / `def numberCount(self, a: int, b: int) -> int`?